In [2]:
import cv2
import mediapipe as mp
import random
import time
import warnings 
warnings.filterwarnings("ignore", category=UserWarning)

# Mediapipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_draw = mp.solutions.drawing_utils

# Start webcam
cap = cv2.VideoCapture(0)

player_score = 0
computer_score = 0

choices = ["rock", "paper", "scissors"]

round_start = time.time()
round_delay = 5
countdown_done = False


def get_winner(player, computer):

    if player == computer:
        return "Draw"

    if (player == "rock" and computer == "scissors") or \
       (player == "paper" and computer == "rock") or \
       (player == "scissors" and computer == "paper"):
        return "Player"

    return "Computer"


def detect_move(hand_landmarks):

    lm = hand_landmarks.landmark

    fingers = []

    if lm[8].y < lm[6].y:
        fingers.append(1)
    else:
        fingers.append(0)

    if lm[12].y < lm[10].y:
        fingers.append(1)
    else:
        fingers.append(0)

    if lm[16].y < lm[14].y:
        fingers.append(1)
    else:
        fingers.append(0)

    if lm[20].y < lm[18].y:
        fingers.append(1)
    else:
        fingers.append(0)

    total = sum(fingers)

    if total == 0:
        return "rock"

    elif total == 2:
        return "scissors"

    elif total >= 4:
        return "paper"

    return None


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb)

    player_move = None

    if results.multi_hand_landmarks:

        for hand in results.multi_hand_landmarks:

            mp_draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

            player_move = detect_move(hand)

    current_time = time.time()
    elapsed = current_time - round_start

    countdown = int(3 - elapsed)

    if countdown >= 0:

        cv2.putText(frame, str(countdown + 1), (300,200),
                    cv2.FONT_HERSHEY_SIMPLEX,5,(0,0,255),10)

    else:

        if not countdown_done:

            computer_move = random.choice(choices)

            if player_move:

                winner = get_winner(player_move, computer_move)

                if winner == "Player":
                    player_score += 1

                elif winner == "Computer":
                    computer_score += 1

            else:
                winner = "No hand detected"

            countdown_done = True

        cv2.putText(frame, f"Player Move: {player_move}", (20,50),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

        cv2.putText(frame, f"Computer Move: {computer_move}", (20,90),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)

        cv2.putText(frame, f"Winner: {winner}", (20,130),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,255),2)

        if elapsed > round_delay:
            round_start = time.time()
            countdown_done = False

    # Score board
    cv2.putText(frame, f"Player Score: {player_score}", (20,400),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    cv2.putText(frame, f"Computer Score: {computer_score}", (20,440),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)


    # Game end condition
    if player_score == 10 or computer_score == 10:

        if player_score > computer_score:
            final_text = "PLAYER WINS THE GAME!"
        else:
            final_text = "COMPUTER WINS THE GAME!"

        while True:

            ret, frame = cap.read()
            frame = cv2.flip(frame,1)

            cv2.putText(frame, final_text, (70,200),
                        cv2.FONT_HERSHEY_SIMPLEX,1.5,(0,255,0),4)

            cv2.putText(frame, f"Player: {player_score}", (200,300),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255),2)

            cv2.putText(frame, f"Computer: {computer_score}", (200,350),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255),2)

            cv2.imshow("Game Over", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        break


    cv2.imshow("Rock Paper Scissors Game", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()
